# v4 파이프라인 전체 — 질문 하나가 Step 1~4 를 지나며 무엇이 되는가

질문을 넣으면 단계마다 **입력·파라미터·출력**을 표로 보여 주고, 마지막에 **찾은 View(잘라 낸 표)** 를 그대로 보여 준다.
계산은 API·CLI 와 같은 backend 함수, 화면 출력은 `notebooks/v4/e2e_helpers.py`.

| 단계 | 하는 일 | 함수 (`backend/…`) | Swagger |
|---|---|---|---|
| ⓪ 준비 | 표 색인 · CSV 값 사전 | `rag/ingestion/table_text` · `rag/csv_index` | `POST /api/v1/rag/chunk-and-index` · `/api/v1/csv-index/build` |
| ① 쿼리 (Step 2) | 재작성 → 동의어·접지 → 시점 규칙·되묻기 → 서브쿼리 → 검색 문장·view_bundle | `rag/query_analyzer/v4.analyze` | `POST /api/v1/rag/v4/analyze` · `/clarify` |
| ② 검색 (Step 3) | 검색 단위마다 구성별 후보 표 (단계별 순위) | `rag/retrieval/factory.get_pipeline(…).search` | `POST /api/v1/retrieval-lab/search` · `/compare` |
| ③ View 묶음 (Step 4) | 단위마다 상위 표를 슬롯대로 잘라 View | `rag/view.build_viewset` | `POST /api/v1/rag/v4/views` |
| ④ 충돌·선택 (Step 4) | 같은 교차점 값 비교 → 선택지 → 선택 | `detect` · `apply_selections` | `POST /api/v1/rag/v4/views/select` |
| ⑤ 최종 View | 답변 모듈로 넘길 Record (+ 선택: HCX 답변) | `agent/light_agent.answer_from_records` | `POST /api/v1/agent/answer` |

**쓰는 법**: `설정` 셀에서 질문·검색 구성·되묻기/선택 방식을 바꾸고 위에서부터 실행 →
`summary_table(R)` 로 전체, `show(R[질문])` 으로 한 질문을 단계별로.

> 저장소 루트에서 실행. 실제 데이터·HCX 를 쓰면 질문마다 HCX 1회(쿼리 분석) + 임베딩(캐시).
> 데이터·HCX 없이 구조만 보려면 `V4_NB_FAKE=1` 환경변수 (가짜 데이터 + HCX 대역 + 충돌 재현용 표).

## 0. 환경

In [ ]:
import os, sys, json, time
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parents[1]   # notebooks/v4 에서 열어도
sys.path[:0] = [str(ROOT), str(ROOT / "notebooks" / "v4")]; os.chdir(ROOT)
USE_FAKE = os.getenv("V4_NB_FAKE", "0") == "1"
if USE_FAKE:                                   # 가짜 KOSIS 데이터 + 표 15개 + 충돌 재현 표 2개, HCX 대역
    import tempfile
    from tests import fake_kosis
    from tests.v4.test_retrieval_lab import _extra_tables
    from tests.v4.test_view_v4 import build_conflict_data
    tmp = Path(tempfile.mkdtemp()) / "storage"
    fake_kosis.TABLES[:] = fake_kosis.TABLES + _extra_tables()
    build_conflict_data(tmp / "kosis_data")
    for k in ("KOSIS_DATA_DIR", "KOSIS_DATA", "VIEW_OUT_DIR", "TABLE_INDEX_DIR", "VIEW_EMBED_CACHE", "CSV_INDEX_DIR", "RUNS_DIR"):
        os.environ[k] = ""
    os.environ["STORAGE_DIR"], os.environ["HCX_API_KEY"] = str(tmp), "test"
    import backend.core.llm as llm, backend.rag.retrieval.dense as dense
    llm._post, dense._post = fake_kosis.fake_hcx_post, fake_kosis.fake_dense_post

import pandas as pd
from IPython.display import display
pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_rows", 60)
from backend.core.config import KOSIS_DATA_DIR, TABLE_INDEX_DIR, CSV_INDEX_DIR, RUNS_DIR
from e2e_helpers import run_one, show, summary_table, choose, show_options, show_records, ok_configs
print("데이터:", KOSIS_DATA_DIR, "\n표 색인:", TABLE_INDEX_DIR, "\nCSV 값 사전:", CSV_INDEX_DIR, "\n실행 기록:", RUNS_DIR,
      "\n가짜 데이터:", USE_FAKE)

## ⓪ 준비 — 표 색인 · CSV 값 사전

이미 있으면 건너뜀. `REBUILD=True` 면 다시 만든다 (CSV 값 사전은 바뀐 표만 다시 스캔).

In [ ]:
REBUILD = False
from backend.rag.ingestion import meta_profile, table_text
from backend.rag.csv_index import build_index, load_index
if REBUILD or USE_FAKE or not (Path(TABLE_INDEX_DIR) / "tokens.json").exists():
    meta_profile.build_all(); table_text.build_index(progress=print)
if REBUILD or USE_FAKE or not (Path(CSV_INDEX_DIR) / "values.parquet").exists():
    build_index(scope="all", log=print)
idx = load_index()
print("표 색인:", json.loads((Path(TABLE_INDEX_DIR) / "meta.json").read_text(encoding="utf-8")))
print("CSV 값 사전:", {k: idx.summary.get(k) for k in ("n_tables", "n_values", "by_slot")})

## 설정 — 여기만 바꿔서 실험

| 키 | 뜻 |
|---|---|
| `QUESTIONS` | 실험 질문 (가짜 데이터면 아래 예시가 충돌·추이·성별연령·서브쿼리·되묻기·범위 밖을 하나씩 보여 줌) |
| `today` | 기준 날짜 ('작년'·'최근' 해석) |
| `query_overrides` | `configs/query_v4.yaml` 값 일부 (예: `{"base_year": 2025}`, `{"conflict_rel_tol": 0.01}`, `{"view_keep_unusable": False}`) |
| `retrieval_configs` | ② 에서 나란히 볼 검색 구성 (`configs/retrieval/*.yaml`). 설치 안 된 구성은 자동으로 뺌 |
| `view_config` | ③ View 를 만들 때 쓸 검색 구성 (`None` = 기존 검색 = baseline) |
| `n_views` | 단위마다 View 를 만들어 비교할 표 수 |
| `clarify` | 되묻기 응답: `"first"`(첫 후보) · `"ask"`(input) · `{질문: {슬롯: [값]}}` · `None`(멈춤) |
| `select` | 충돌 선택: `"first"` · `"ask"` · `{질문: "opt-2"}` · `None`(선택 대기로 두고 아래 '선택' 셀에서) |

In [ ]:
QUESTIONS = [
    "2025년 대전 인구 수",                     # 같은 교차점 값이 다른 표 → 충돌·선택 (가짜 데이터)
    "최근 5년간 대전의 인구 변화 추이",          # 추이 (시점 규칙)
    "작년 대구의 남자 20대 인구 수",             # 성별·연령 슬롯
    "2025년 서울 출생아 수랑 사망자 수",         # 함께 가진 표 없음 → 서브쿼리 2개
    "요즘 1인 가구는 몇 가구야?",               # 지역 없음 → 되묻기 + '요즘' → 2026
    "합계출산율이 무슨 뜻이야?",                # 범위 밖
]
CFG = {
    "today": "2026-11-09",
    "query_overrides": {"csv_scope": "all"} if USE_FAKE else {},
    "retrieval_configs": ["baseline", "bm25_only", "kiwi_lexicon", "hybrid_rrf_hcx"],
    "view_config": None,
    "n_views": 3,
    "clarify": "first",
    "select": None,
    "max_view_rows": 15,
}
RUN_ANSWER = False                 # ⑤ 에서 HCX 답변까지 (실제 HCX 키 필요)

## 실행 — 질문 전체

In [ ]:
R = {}
for q in QUESTIONS:
    print("▶", q)
    R[q] = run_one(q, CFG)
summary_table(R)

## 질문별 자세히 — 번호를 바꿔 가며 (`show(R[질문])`)

In [ ]:
show(R[QUESTIONS[0]], CFG)

In [ ]:
show(R[QUESTIONS[3]], CFG)

In [ ]:
show(R[QUESTIONS[4]], CFG)

## 선택 — 충돌이 남은 질문 고르기

`choose(R, 질문, option_id)` → 선택을 반영하고 최종 View 를 보여 준다.
(Swagger: `POST /api/v1/rag/v4/views/select` 에 `{selection_token, option_id}` + 이전 응답을 `previous` 로)

In [ ]:
pending = [q for q, r in R.items() if (r.get("viewset") or {}).get("pending")]
print("선택 대기:", pending)
if pending:
    show_options(R[pending[0]]["viewset"]["pending"][0])

In [ ]:
if pending:
    choose(R, pending[0], "opt-1", CFG)          # ← 원하는 opt-n 으로
summary_table(R)

## ⑤ (선택) 답변까지 — HCX 1회

`RUN_ANSWER = True` 일 때만. 최종 View 로 `answer_from_records` (= `POST /api/v1/agent/answer`).

In [ ]:
if RUN_ANSWER and not USE_FAKE:
    from IPython.display import Markdown
    from backend.agent.light_agent import answer_from_records
    for q, r in R.items():
        vs = r.get("viewset") or {}
        if vs.get("status") == "ok" and vs.get("records"):
            r["answer"] = out = answer_from_records(q, vs["records"])
            display(Markdown(f"**{q}**\n\n{out.get('answer')}\n\n_검증 안 된 숫자: {out.get('unverified_numbers')}_"))
else:
    print("RUN_ANSWER=False (또는 가짜 데이터) — 건너뜀")

## 저장 · 검색 구성 비교 보고서

- `R` 전체를 `storage/runs/<시각>_nb_e2e/results.json` 으로 (나중에 다시 보거나 팀원과 공유)
- 이 질문들의 검색 단위로 Step 3 비교 보고서(`report.html`)·판정표(`review.csv`)도 만든다

In [ ]:
out_dir = Path(RUNS_DIR) / f"{time.strftime('%Y%m%d_%H%M%S')}_nb_e2e"
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "results.json").write_text(json.dumps(R, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
summary_table(R).to_csv(out_dir / "summary.csv", index=False, encoding="utf-8-sig")
print("저장:", out_dir)

from backend.rag.retrieval import lab
qs = [{"id": f"Q{i}-{u['unit_id']}", "question": q, "search_text": u["search_query"], "filters": u["filters"], "prep": False}
      for i, (q, r) in enumerate(R.items(), 1) for u in r.get("retrieval") or []]
if qs:
    cmp = lab.run_compare(lab.load_queries(qs), ok_configs(CFG["retrieval_configs"]), top_k=5)
    print("비교 보고서:", cmp["files"]["html"], "\n판정표:", cmp["files"]["review"])
    display(pd.DataFrame(cmp["summary"]).T)